# The order flow imbalance

Section 1.4 of the lecture notes, at the keyboard, from Assumption 1.4.1 to Proposition 1.4.8.
We build the idealised book as an object, let a six-type order flow act on it, and check each
statement on what the package's own book and session record: what one event contributes, what
a window sums, and how the sum moves the mid-price.

Sections 1 to 5 go with §1.4.1, on the order flow imbalance; section 6 goes with
Proposition 1.4.8 of §1.4.2.  Prices are integer counts of ticks throughout, so $\tau = 1$ in
every formula below.

In [ ]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from unito26.lob import config
from unito26.lob.hawkes import ExponentialHawkes
from unito26.lob.idealised import IdealisedFlow, UniformDepth, idealised_book
from unito26.lob.messages import (
    BUY, SELL, GridDepth, PriceUnit, ReportedDepth, Window, limit_order, market_order,
)
from unito26.lob.orderbook import AggregateBook
from unito26.lob.session import MarketSession, window_start
from unito26.lob.simulate import EventJournal, EventType
from unito26.lob.statistics import SessionStatistics
from unito26.lob.visualization import (
    BID_COLOUR, INK, MICRO_COLOUR, MID_COLOUR, MUTED, PALETTE,
    ascii_ladder, book_figure, describe_message, snapshots_figure, use_template,
)

use_template()

TICK_SIZE = 1
UNIFORM_DEPTH = UniformDepth(5)
SPEC = SessionStatistics((GridDepth(1),), (), (1,))
FLOW_COLOUR = PALETTE[4]

NAMES = {
    EventType.MARKET_BUY: "market buy",
    EventType.MARKET_SELL: "market sell",
    EventType.LIMIT_BUY: "limit buy",
    EventType.LIMIT_SELL: "limit sell",
    EventType.WITHDRAW_BUY: "withdraw bid",
    EventType.WITHDRAW_SELL: "withdraw ask",
}
PRESSURE = np.array([event.pressure for event in EventType])

## 1. The idealised book

Assumption 1.4.1 fixes $\bar S > 0$ and asks four things:

- *i.* no event occurs behind the best quote of its side;
- *ii.* no order reaches past the quote it acts on;
- *iii.* every queue holds at most $\bar S$, and every queue behind the best bid and the best
  ask holds exactly $\bar S$;
- *iv.* once the size at a touch reaches $\bar S$, the next limit order on that side arrives
  one tick inside the spread.

Part *iii.* is a property of the book; the other three are properties of the events that act
on it.  `idealised_book` builds a book with part *iii.* from its two touches.  The assumption
puts a queue of $\bar S$ on every tick behind a touch, without end, and the book stops after a
finite number of levels; the flow below stays far from the last of them.

In [ ]:
OPENING = {BUY: (1000, 3), SELL: (1020, 2)}
LEVELS = GridDepth(60)

book = idealised_book(UNIFORM_DEPTH, OPENING, LEVELS)
book_figure(book, depth=8, title="The opening book, eight levels a side")

Behind each touch every level holds $\bar S$, and each touch holds less.  The spread is wide on
purpose: a limit order at a full touch needs a free tick inside it.  On the object, part *iii.* is
three checks a side.

In [ ]:
def part_iii(book, direction):
    '''Part iii on one side of the book, as three checks.'''
    levels = book.levels_map(direction)
    prices = sorted(levels, reverse=direction == BUY)
    touch, behind = prices[0], prices[1:]
    return {
        "the touch holds 1 to Sbar": 1 <= levels[touch] <= UNIFORM_DEPTH,
        "every level behind holds Sbar": all(levels[price] == UNIFORM_DEPTH for price in behind),
        "no tick behind the touch is empty":
            behind == [touch - direction * k for k in range(1, len(prices))],
    }


print(f"bid touch {book.best_bid_size} @ {book.best_bid_price}, "
      f"ask touch {book.best_ask_size} @ {book.best_ask_price}, "
      f"{len(book.levels_map(BUY))} levels a side")
pd.DataFrame({"bid": part_iii(book, BUY), "ask": part_iii(book, SELL)})

The events come from `IdealisedFlow`.  It takes the six-type Hawkes process of Definition 1.4.6,
with the package's example parametrisation, and places each event as an order of one share: a
market order at the opposite touch, a withdrawal at its own touch, and a limit order at its own
touch, or one tick inside the spread when that touch holds $\bar S$.  The sizes are units because
part *ii.* needs them: an order of two shares can overshoot the cap, or reach past a touch that
holds one.

Some events cannot be placed so: a limit order at a full touch when the spread is one tick, and
a market order or a withdrawal that would empty a side.  The flow raises `AssumptionBreach` on
them rather than move or drop them.

We fold the flow into the opening book by hand.  Each message is checked against parts *i.*,
*ii.* and *iv.* on the book it arrives at, and the book it leaves against part *iii.*  The journal
records each event as the flow places it, so its last entry is the type behind the message in
hand.

In [ ]:
SEED = 11
HORIZON = 60.0
MARKET = (EventType.MARKET_BUY, EventType.MARKET_SELL)
LIMIT = (EventType.LIMIT_BUY, EventType.LIMIT_SELL)


def example_flow():
    '''The idealised flow of the example regime, on a fixed seed: the same events every call.'''
    hawkes = ExponentialHawkes(config.example_order_flow_params(), rng=SEED)
    return IdealisedFlow(hawkes, UNIFORM_DEPTH)


def placed_as_assumed(book, message, event):
    '''Parts i, ii and iv for one message, against the book it arrives at.'''
    side = -event.direction if event in MARKET else event.direction
    touch = book.best_price(side)
    if event in LIMIT:
        full = book.size_at(side, touch) == UNIFORM_DEPTH
        return message.price == (touch + side if full else touch)
    return message.price == touch and message.size <= book.size_at(side, touch)


book = idealised_book(UNIFORM_DEPTH, OPENING, LEVELS)
journal = EventJournal.empty()
sizes, spreads, fewest_levels = [], [], []
for message in example_flow().stream(book, HORIZON, journal):
    assert placed_as_assumed(book, message, EventType(journal.types[-1]))
    book.apply(message, record=False)
    assert all(all(part_iii(book, direction).values()) for direction in (BUY, SELL))
    sizes.append(message.size)
    spreads.append(book.spread)
    fewest_levels.append(min(len(book.levels_map(BUY)), len(book.levels_map(SELL))))

print(f"{len(journal):,} messages in {HORIZON:.0f} s, of sizes {sorted(set(sizes))}")
print("parts i, ii and iv held for every message, and part iii after every one")
print(f"spread between {min(spreads)} and {max(spreads)} ticks; "
      f"fewest levels left on a side: {min(fewest_levels)} of {LEVELS}")
events = pd.Series([NAMES[EventType(event)] for event in journal.types])
events.value_counts().reindex(list(NAMES.values())).rename("events")

Every message passed, and so did every state.  The run stayed far from a breach: the spread
stayed many ticks wide and each side kept most of its levels.  Limit orders are the commonest
events and market orders the rarest.

## 2. What each event contributes

Definition 1.4.2 writes $S^{b,1}_{T_n}$ for the size at the best bid *after* the $n$-th event,
so that $S^{b,1}_{T_{n-1}}$ is the size standing just before it, and sets, in (1.4.1),

$$
e_n = \mathbf 1_{\{P^b_{T_n} \ge P^b_{T_{n-1}}\}}\, S^{b,1}_{T_n}
- \mathbf 1_{\{P^b_{T_n} \le P^b_{T_{n-1}}\}}\, S^{b,1}_{T_{n-1}}
- \mathbf 1_{\{P^a_{T_n} \le P^a_{T_{n-1}}\}}\, S^{a,1}_{T_n}
+ \mathbf 1_{\{P^a_{T_n} \ge P^a_{T_{n-1}}\}}\, S^{a,1}_{T_{n-1}} .
$$

On the bid side there are three cases.  If the best bid is unchanged, both indicators fire and
the event contributes the change in the size resting there.  If it improves, only the first
fires, and the event contributes the whole new queue.  If it is cleared, only the second fires,
and the event contributes minus the whole old queue.  The ask side is the same with the signs
reversed.

To see every case we choose the events instead of drawing them.  A scripted source stands where
the Hawkes process stood, yielding one event a second, and `IdealisedFlow` places each as it
would a simulated one.  The book opens with a spread of two ticks; the first five events act on
the bid and the last five on the ask.

In [ ]:
class Script:
    '''Chosen event types, one a second, in the place of the Hawkes process.

    `IdealisedFlow` checks that its source has six types, and reads nothing else of `params`.
    '''

    params = config.example_order_flow_params()

    def __init__(self, types):
        self.script = [(float(second), event) for second, event in enumerate(types, start=1)]

    def events(self, horizon):
        return ((time, event) for time, event in self.script if time <= horizon)


def fold(flow, book, horizon, reported_depth):
    '''The session of `flow` acting on `book`, and the type of the event behind each row.'''
    journal = EventJournal.empty()
    session = MarketSession.from_occupied_levels(
        book, flow.stream(book, horizon, journal), reported_depth, SPEC, PriceUnit(1), True
    )
    return session, np.array(journal.types)


SCRIPT = [
    EventType.LIMIT_BUY, EventType.LIMIT_BUY, EventType.WITHDRAW_BUY,
    EventType.MARKET_SELL, EventType.WITHDRAW_BUY,
    EventType.LIMIT_SELL, EventType.LIMIT_SELL, EventType.MARKET_BUY,
    EventType.WITHDRAW_SELL, EventType.MARKET_BUY,
]

script_book = idealised_book(UNIFORM_DEPTH, {BUY: (1000, 4), SELL: (1002, 4)}, GridDepth(4))
script_opening = script_book.copy()
script_session, script_types = fold(
    IdealisedFlow(Script(SCRIPT), UNIFORM_DEPTH), script_book, float(len(SCRIPT)),
    ReportedDepth(5),
)

The formula needs the touch before and after each event.  The session holds the book after every
event; the touch before the first is the opening book's.  We evaluate (1.4.1) on those touches
by hand, and set it beside the session's own `OrderFlowContribution`.

In [ ]:
def touch(book):
    '''(P^b, S^b, P^a, S^a) of a book.'''
    return book.best_bid_price, book.best_bid_size, book.best_ask_price, book.best_ask_size


def touches(opening, session):
    '''The touch before the first event, then after each: entry n follows event n.'''
    after = session.lobster_book[["BidPrice1", "BidSize1", "AskPrice1", "AskSize1"]]
    return [touch(opening)] + [tuple(row) for row in after.itertuples(index=False)]


def contribution(previous, current):
    '''e_n by (1.4.1), from (P^b, S^b, P^a, S^a) at T_{n-1} and at T_n.'''
    bid_0, bid_size_0, ask_0, ask_size_0 = previous
    bid, bid_size, ask, ask_size = current
    return (
        (bid >= bid_0) * bid_size - (bid <= bid_0) * bid_size_0
        - (ask <= ask_0) * ask_size + (ask >= ask_0) * ask_size_0
    )


MOVES = {0: "unchanged", 1: "improved", -1: "cleared"}


def case(previous, current):
    '''The side the event changed, and what became of that side's best price.'''
    if previous[:2] != current[:2]:
        return "bid " + MOVES[np.sign(current[0] - previous[0])]
    return "ask " + MOVES[np.sign(previous[2] - current[2])]


def queue(price, size):
    return f"{size} @ {price}"


states = touches(script_opening, script_session)
table = pd.DataFrame(
    [
        {
            "event": NAMES[EventType(event)],
            "pressure": EventType(event).pressure,
            "bid before": queue(*states[n - 1][:2]),
            "bid after": queue(*states[n][:2]),
            "ask before": queue(*states[n - 1][2:]),
            "ask after": queue(*states[n][2:]),
            "case": case(states[n - 1], states[n]),
            "e_n, by (1.4.1)": float(contribution(states[n - 1], states[n])),
        }
        for n, event in enumerate(script_types, start=1)
    ],
    index=pd.RangeIndex(1, len(script_types) + 1, name="n"),
)
table.insert(7, "e_n, session", script_session.stats["OrderFlowContribution"].to_numpy())

agree = (table["e_n, session"] == table["e_n, by (1.4.1)"]).iloc[1:]
print("event 1, session:", table["e_n, session"].iloc[0])
print(f"events 2 to {len(table)}: the session and (1.4.1) agree on {agree.sum()} of {len(agree)}")
table

The session and the formula agree wherever the session answers.  On the first row it does not.
$e_n$ reads two consecutive states, and the first row has no predecessor in the session: the fold
holds the opening book and declines to read it, so that recomputing the statistics from the
finished frame, which holds no opening book, gives the same series.  A window that contains the
first row inherits the NaN; section 3 meets such windows.

Every case of (1.4.1) occurs.  Events 2 and 7 open a queue on the one tick inside a spread of
two, and contribute the whole new queue.  Events 3 and 8 empty a touch, by a withdrawal and by a
market order.  The level behind becomes the touch, with $\bar S$ resting on it, and the event
contributes the one share of the old queue, not the $\bar S$ of the new one.

On every row the contribution is one share with the sign of the pressure.  That is
Proposition 1.4.8, and section 6 checks it on a simulated session.  The book after each event:

In [ ]:
books = {"opening": script_opening}
for n, event in enumerate(script_types, start=1):
    books[f"{n}. {NAMES[EventType(event)]}"] = AggregateBook.from_lobster_row(
        script_session.lobster_book.iloc[n - 1], PriceUnit(1), ReportedDepth(5)
    )
snapshots_figure(books, depth=5, title="")

## 3. The order flow imbalance over a window

$\mathrm{OFI}_{t,w}$ sums $e_n$ over the events in $(t-w, t]$, as in (1.4.2).  The session
records it as a column for each window its specification names, one second here, and
`order_flow_imbalance` computes it for any other.  Both take one difference of a cumulative sum.
A third route is the definition, evaluated row by row.

The session folds the flow of section 1 again, from the same seed and the same opening book, so
its rows follow the messages that section 1 checked.

In [ ]:
session, types = fold(
    example_flow(), idealised_book(UNIFORM_DEPTH, OPENING, LEVELS), HORIZON, ReportedDepth(1)
)
assert types.tolist() == journal.types

times = session.lobster_book.index.to_numpy()
contributions = session.stats["OrderFlowContribution"].to_numpy()

WINDOW = 1.0
declared = session.stats["OrderFlowImbalance1"].to_numpy()
imbalance = session.order_flow_imbalance(Window(WINDOW)).to_numpy()
by_definition = np.array(
    [contributions[(times > t - WINDOW) & (times <= t)].sum() for t in times]
)

undefined = np.isnan(imbalance)
print(f"{len(times):,} rows, the messages of section 1")
print("the column and the method agree    :",
      np.array_equal(declared, imbalance, equal_nan=True))
print("the method and the definition agree:",
      np.array_equal(imbalance, by_definition, equal_nan=True))
print(f"undefined on {undefined.sum()} rows, exactly those within w of the first event:",
      np.array_equal(undefined, times < times[0] + WINDOW))

The undefined rows are those whose window reaches back past the first event, and so contains
the first row.

Over the same trailing windows the mid-price changes by $P^m_t - P^m_{t-w}$, where
$P^m_{t-w}$ is the mid-price of the last row at or before $t - w$.  The two series side by side:

In [ ]:
mid = session.stats["MidPrice"].to_numpy()
standing = window_start(times, WINDOW) - 1          # the row whose state stood at t - w
mid_change = np.where(standing >= 0, mid - mid[np.maximum(standing, 0)], np.nan)
print("the mid-price change is undefined on the same rows:",
      np.array_equal(np.isnan(mid_change), undefined))

figure = make_subplots(
    rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.1,
    subplot_titles=(f"Mid-price change over (t - w, t], w = {WINDOW:g} s",
                    f"Order flow imbalance over (t - w, t], w = {WINDOW:g} s"),
)
figure.add_trace(go.Scatter(x=times, y=mid_change, mode="lines", line_shape="hv",
                            line=dict(color=MID_COLOUR, width=1.5), name="mid-price change"),
                 row=1, col=1)
figure.add_trace(go.Scatter(x=times, y=imbalance, mode="lines", line_shape="hv",
                            line=dict(color=FLOW_COLOUR, width=1.5), name="OFI"),
                 row=2, col=1)
figure.update_yaxes(title_text="ticks", row=1, col=1)
figure.update_yaxes(title_text="shares", row=2, col=1)
figure.update_xaxes(title_text="session time (s)", row=2, col=1)
figure.update_layout(height=560, showlegend=False, hovermode="x unified")
figure

The two series rise and fall together.  The imbalance counts shares and the mid-price change
counts half ticks; section 4 states the exact relation between them.

## 4. The mid-price moves with the flow

Proposition 1.4.3: under the assumption,

$$
\Delta P^m_{t,w} = \frac{\tau}{2\bar S}\,\mathrm{OFI}_{t,w} + \varepsilon_{t,w},
\qquad
\varepsilon_{t,w} = -\frac{\tau}{2\bar S}\Big[\big(S^{b,1}_t - S^{b,1}_{t-w}\big)
- \big(S^{a,1}_t - S^{a,1}_{t-w}\big)\Big],
\qquad
|\varepsilon_{t,w}| \le \tau .
$$

Multiplied by $2\bar S/\tau$ it becomes an identity between integers,

$$
\frac{2\bar S}{\tau}\,\Delta P^m_{t,w} = \mathrm{OFI}_{t,w}
- \big(S^{b,1}_t - S^{b,1}_{t-w}\big) + \big(S^{a,1}_t - S^{a,1}_{t-w}\big),
$$

since the mid-price moves by half ticks and sizes are whole shares.  In that form it can be
checked with no tolerance at all.

We sample windows uniformly at random, as pairs of rows.  The window from row $i$ to row $j > i$
runs from the time of row $i$, excluded, to the time of row $j$, and holds the events of rows $i+1$
to $j$; so the first row, whose contribution is undefined, is never inside one.  Over such a window
the imbalance is a difference of the cumulative sum of $e_n$, the reduction the session itself
performs.

In [ ]:
bid_price = session.lobster_book["BidPrice1"].to_numpy()
ask_price = session.lobster_book["AskPrice1"].to_numpy()
bid_size = session.lobster_book["BidSize1"].to_numpy()
ask_size = session.lobster_book["AskSize1"].to_numpy()
cumulative = np.nancumsum(contributions)

rng = np.random.default_rng(2026)
pairs = np.sort(rng.integers(0, len(times), size=(2000, 2)), axis=1)
first, last = pairs[pairs[:, 0] < pairs[:, 1]].T

window_ofi = cumulative[last] - cumulative[first]
delta_mid = mid[last] - mid[first]
delta_bid_size = bid_size[last] - bid_size[first]
delta_ask_size = ask_size[last] - ask_size[first]
slope = TICK_SIZE / (2 * UNIFORM_DEPTH)
residual = -slope * (delta_bid_size - delta_ask_size)

in_integers = (
    2 * UNIFORM_DEPTH * delta_mid / TICK_SIZE - (window_ofi - delta_bid_size + delta_ask_size)
)
as_stated = delta_mid - (slope * window_ofi + residual)
assert np.all(in_integers == 0)
assert np.all(np.abs(residual) <= TICK_SIZE)

lengths = times[last] - times[first]
print(f"{len(window_ofi):,} windows, "
      f"from {lengths.min() * 1000:.1f} ms to {lengths.max():.1f} s long")
print("slope tau / (2 Sbar)                 :", slope)
print("identity in integers, largest error  :", np.abs(in_integers).max())
print(f"identity as stated, largest error    : {np.abs(as_stated).max():.1e}")
print("largest |residual|, ticks            :", np.abs(residual).max())

In integers the identity holds with no error at all.  Written as the notes state it, it holds up
to the rounding of floating-point arithmetic: the slope printed above has no finite binary
expansion, so the flow term carries a rounding error that the integer form does not.

The largest residual is $\tau(\bar S - 1)/\bar S$, reached when one touch goes from one share to
$\bar S$ over the window and the other from $\bar S$ to one.  It stays below $\tau$ because a touch
never holds less than one share.

The same windows, drawn:

In [ ]:
def identity_figure(low, high, title):
    '''The line tau OFI / (2 Sbar) for OFI in [low, high], and the band of one tick either side.'''
    x = np.array([low, high])
    line = slope * x
    return go.Figure(
        [
            go.Scatter(x=np.r_[x, x[::-1]], y=np.r_[line + TICK_SIZE, line[::-1] - TICK_SIZE],
                       mode="lines", fill="toself", fillcolor="rgba(137, 135, 129, 0.18)",
                       line=dict(width=0), hoverinfo="skip", name="one tick either side"),
            go.Scatter(x=x, y=line, mode="lines", line=dict(color=INK, width=1.5),
                       name="τ OFI / (2 S̄)"),
        ],
        layout=dict(title=title, height=520,
                    xaxis=dict(title="order flow imbalance over the window (shares)",
                               range=[low, high]),
                    yaxis=dict(title="mid-price change over the window (ticks)",
                               range=[line[0] - 1.5 * TICK_SIZE, line[1] + 1.5 * TICK_SIZE]),
                    legend=dict(orientation="h", y=1.08, x=0)),
    )


figure = identity_figure(window_ofi.min() - 2, window_ofi.max() + 2,
                         "The mid-price change against the imbalance, over random windows")
figure.add_trace(go.Scatter(x=window_ofi, y=delta_mid, mode="markers", name="a window",
                            marker=dict(color=MID_COLOUR, size=7, opacity=0.35)))
figure

Every window lies inside the band, whose width does not depend on the imbalance: over every
window the mid-price change is the flow term to within a tick.

### The position of the bid, in shares

Exercise 1.4.4 proves the proposition through the position of the bid in shares,

$$
S^{b,1}_s + \bar S\,\frac{P^b_s - P^b_{t-w}}{\tau}, \qquad t - w \le s \le t,
$$

which each event moves by its own signed bid-side flow: $+q$ for a limit buy, $-q$ for a
withdrawal from the bid or a market sell, and nothing for an event on the ask side.  We draw it
over the stretch of consecutive events in which the best bid travels furthest.

In [ ]:
BID_SIDE_FLOW = {EventType.LIMIT_BUY: 1, EventType.WITHDRAW_BUY: -1, EventType.MARKET_SELL: -1}
ASK_SIDE_FLOW = {EventType.LIMIT_SELL: 1, EventType.WITHDRAW_SELL: -1, EventType.MARKET_BUY: -1}
STRETCH = 120

i = int(np.argmax(np.abs(bid_price[STRETCH:] - bid_price[:-STRETCH])))
j = i + STRETCH
rows = np.arange(i, j + 1)
inside = [EventType(event) for event in types[i + 1 : j + 1]]

bid_position = bid_size[rows] + UNIFORM_DEPTH * (bid_price[rows] - bid_price[i]) / TICK_SIZE
ask_position = ask_size[rows] - UNIFORM_DEPTH * (ask_price[rows] - ask_price[i]) / TICK_SIZE
assert np.array_equal(np.diff(bid_position), [BID_SIDE_FLOW.get(event, 0) for event in inside])
assert np.array_equal(np.diff(ask_position), [ASK_SIDE_FLOW.get(event, 0) for event in inside])

bid_travel = bid_position[-1] - bid_position[0]
ask_travel = ask_position[-1] - ask_position[0]
print(f"window ({times[i]:.3f} s, {times[j]:.3f} s]: {STRETCH} events, "
      f"{sum(event in BID_SIDE_FLOW for event in inside)} of them on the bid side")
print(f"best bid from {bid_price[i]} to {bid_price[j]}; "
      f"every step of the position is the event's own bid-side flow")
bid_ticks = bid_price[rows] - bid_price[i]
assert np.array_equal(bid_ticks, np.ceil(bid_position / UNIFORM_DEPTH) - 1)
print("on every row the best bid is ceil(position / Sbar) - 1 ticks from its start")
print(f"change of the bid position {bid_travel:+.0f}, of the ask position {ask_travel:+.0f}, "
      f"OFI over the window {cumulative[j] - cumulative[i]:+.0f}")
assert bid_travel - ask_travel == cumulative[j] - cumulative[i]

figure = make_subplots(
    rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.1, row_heights=[0.62, 0.38],
    subplot_titles=("The position of the bid, in shares",
                    "The best bid, in ticks from its value at the start of the window"),
)
figure.add_trace(go.Scatter(x=times[rows], y=bid_position, mode="lines", line_shape="hv",
                            line=dict(color=BID_COLOUR, width=2), name="position"),
                 row=1, col=1)
lowest = UNIFORM_DEPTH * int(np.floor(bid_position.min() / UNIFORM_DEPTH))
highest = UNIFORM_DEPTH * int(np.ceil(bid_position.max() / UNIFORM_DEPTH))
for multiple in range(lowest, highest + 1, UNIFORM_DEPTH):
    figure.add_hline(y=multiple, line=dict(color=MUTED, width=1, dash="dash"), row=1, col=1)
figure.add_trace(go.Scatter(x=times[rows], y=bid_ticks, mode="lines",
                            line_shape="hv", line=dict(color=BID_COLOUR, width=2),
                            name="best bid"),
                 row=2, col=1)
figure.update_yaxes(title_text="shares", dtick=UNIFORM_DEPTH, row=1, col=1)
figure.update_yaxes(title_text="ticks", dtick=1, row=2, col=1)
figure.update_xaxes(title_text="session time (s)", row=2, col=1)
figure.update_layout(height=600, showlegend=False, hovermode="x unified")
figure

Each event on the bid side moves the position by one share, and no other event moves it.  The
dashed lines are the multiples of $\bar S$, where the touch is full.  The best bid steps up
exactly when the position rises past one of them, as a limit buy at a full touch opens a queue a
tick higher, and down exactly when the position falls back onto one, as the last share at the
touch leaves and uncovers a full queue a tick lower.  Equivalently, a position of $x$ shares puts
the best bid $\lceil x/\bar S\rceil - 1$ ticks from its start, as the third line printed above
says.

The ask has the mirror image, $S^{a,1}_s - \bar S\,(P^a_s - P^a_{t-w})/\tau$, which each event
moves by its signed ask-side flow.  Summed over the window, the bid-side flows less the ask-side
flows are the imbalance, as the last line printed above says.  Written out in the sizes and the
prices at the two ends of the window, that sum is the identity.

## 5. The residual is the flow that went into the queues

Remark 1.4.5: on a window over which neither quote moves, $\Delta P^m_{t,w} = 0$ while
$\mathrm{OFI}_{t,w} = \big(S^{b,1}_t - S^{b,1}_{t-w}\big) - \big(S^{a,1}_t - S^{a,1}_{t-w}\big)$,
so $\varepsilon_{t,w} = -\tau\,\mathrm{OFI}_{t,w}/(2\bar S)$ and the two terms of the identity
cancel.

The quotes stand still between two events that move one of them.  We cut the session at every
such event, and take each stretch between two cuts as a window.

In [ ]:
moved = np.r_[True, (np.diff(bid_price) != 0) | (np.diff(ask_price) != 0)]
cut = np.flatnonzero(moved)
first_still, last_still = cut, np.r_[cut[1:] - 1, len(times) - 1]
nonempty = last_still > first_still
first_still, last_still = first_still[nonempty], last_still[nonempty]

still_ofi = cumulative[last_still] - cumulative[first_still]
still_queues = (
    (bid_size[last_still] - bid_size[first_still]) - (ask_size[last_still] - ask_size[first_still])
)
assert np.all(mid[last_still] == mid[first_still])
assert np.array_equal(still_ofi, still_queues)

print(f"{len(still_ofi)} windows over which neither quote moves; on every one "
      "the mid-price change is zero and 2 Sbar eps / tau = -OFI")
print(f"the imbalance is non-zero on {np.count_nonzero(still_ofi)} of them, "
      f"and at most {np.abs(still_ofi).max():.0f} shares in size")
values, counts = np.unique(still_ofi, return_counts=True)
print("windows by imbalance:", dict(zip(values.astype(int).tolist(), counts.tolist())))

reach = 4 * UNIFORM_DEPTH
figure = identity_figure(-reach, reach, "Windows over which neither quote moves")
figure.add_trace(go.Scatter(x=window_ofi, y=delta_mid, mode="markers",
                            name="random windows of section 4",
                            marker=dict(color=MUTED, size=6, opacity=0.25)))
figure.add_trace(go.Scatter(x=values, y=np.zeros_like(values), mode="markers",
                            name="neither quote moves", customdata=counts,
                            hovertemplate="OFI %{x}: %{customdata} windows<extra></extra>",
                            marker=dict(color=MICRO_COLOUR, size=6 + 2 * np.sqrt(counts),
                                        line=dict(color="white", width=1))))
figure

On these windows size joined or left the two touches without moving either quote.  The flow term
assigns the mid-price a move it did not make, and the residual cancels it exactly.  On the
scatter these windows lie on the horizontal axis, inside the band, and the imbalance on them is at
most $2(\bar S - 1)$ in size, since it is the difference of two changes of touch sizes that stay
between one share and $\bar S$.  The area of each marker grows with the number of windows at that
imbalance.

The residual is bounded by a tick however long the window, while the flow term is not.  The random
windows of section 4, by length:

In [ ]:
def largest(values):
    return values.abs().max()


by_length = pd.DataFrame(
    {
        "window length (s)": pd.cut(lengths, [0, 1, 4, 16, HORIZON]),
        "flow term": slope * window_ofi,
        "residual": residual,
    }
).groupby("window length (s)", observed=True)
by_length.agg(**{
    "windows": ("residual", "size"),
    "flow term, sd": ("flow term", "std"),
    "flow term, largest": ("flow term", largest),
    "residual, sd": ("residual", "std"),
    "residual, largest": ("residual", largest),
}).round(2)

The flow term spreads out as the window lengthens; the residual has the same spread at every
length and never reaches a tick.  Over the shortest windows the two are of one size, and the
identity says little.  Over the longest the flow term dominates, which is what makes the identity
useful.

## 6. The contribution is the signed size

Proposition 1.4.8: under the assumption $e_n = \varpi_{E_n} q_n$, and so

$$
\mathrm{OFI}_{t,w} = \sum_{n\,:\,t-w < T_n \le t} \varpi_{E_n} q_n .
$$

The session computes $e_n$ from the touches and never sees a type; the pressures are read off the
types and never see the book.  On the session of section 3, with the sizes of the messages that
section 1 checked:

In [ ]:
signed_sizes = PRESSURE[types] * np.array(sizes)
defined = ~np.isnan(contributions)
assert np.array_equal(contributions[defined], signed_sizes[defined])
print(f"e_n = pressure x size on all {defined.sum():,} rows where e_n is defined; "
      f"undefined rows: {np.flatnonzero(~defined).tolist()}")

by_pressure = np.array([signed_sizes[(times > t - WINDOW) & (times <= t)].sum() for t in times])
assert np.array_equal(imbalance[~undefined], by_pressure[~undefined])
print(f"OFI over w = {WINDOW:g} s equals the sum of pressure x size over the window "
      f"on all {(~undefined).sum():,} rows where it is defined")

Under the assumption the imbalance is a functional of the marked point process: the signed sum of
the sizes of the events in the window, whatever the book looked like.

Part *ii.* is what that costs.  On a general book an order may reach past the touch, and $e_n$
still reads the touch alone.  The book below is not idealised: its queues differ, and by the time
the last order arrives the best ask holds less than it takes.  For a submission, limit or market,
the pressure is its direction.

In [ ]:
general = AggregateBook.from_levels(
    {1000: 150, 999: 250, 998: 300}, {1002: 220, 1003: 300, 1004: 250}
)
general_opening = general.copy()
print(ascii_ladder(general_opening))

ORDERS = [
    limit_order(1.0, 100, 1000, BUY),
    market_order(2.0, 100, BUY),
    market_order(3.0, 200, BUY),
]
general_session = MarketSession.from_occupied_levels(
    general, ORDERS, ReportedDepth(1), SPEC, PriceUnit(1), True
)
general_states = touches(general_opening, general_session)
pd.DataFrame(
    {
        "order": [describe_message(order) for order in ORDERS],
        "pressure x size": [order.direction * order.size for order in ORDERS],
        "traded": general_session.trades["Volume"].to_numpy(),
        "ask before": [queue(*state[2:]) for state in general_states[:-1]],
        "ask after": [queue(*state[2:]) for state in general_states[1:]],
        "e_n, session": general_session.stats["OrderFlowContribution"].to_numpy(),
        "e_n, by (1.4.1)": [float(contribution(previous, current))
                            for previous, current in zip(general_states, general_states[1:])],
    },
    index=pd.RangeIndex(1, len(ORDERS) + 1, name="n"),
)

The first row is the session's opening row again, undefined in the session, and the formula
credits the limit buy with its own size.  The first market buy fits inside the touch and
registers its own size, as part *ii.* requires.  The second takes the whole best ask and part of
the level behind it, and registers only the queue it emptied: the shares it took at the next
price are invisible to $e_n$.  The contribution is then the size the touch held, whatever the size
of the order, and the imbalance is no longer a functional of the order flow alone.